# Extraction OEDI — timeseries & météo statique
Deux extracteurs depuis le data lake public OEDI (ResStock 2025, AMY2018) :
- **Partie A** — timeseries par bâtiment (35 040 pas de 15 min × 192 colonnes).
- **Partie B** — météo statique par comté (agrégats climatiques : HDD, CDD, GHI, T design, vent, humidité), joignables sur `in.county`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

# Partie A — timeseries par bâtiment
OEDI_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/'
    'timeseries_individual_buildings/by_state/upgrade=0'
)

# Partie B — météo par comté (même release)
WEATHER_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/weather'
)

## 1bis. Reconstruction des consignes de thermostat

Les consignes (chauffage + clim) sont des schedules **non stochastiques** : absentes (colonnes
nulles) du time series, mais entièrement déterministes à partir de 4 caractéristiques statiques
(`metadata_clean.parquet`) + le masque horaire de `options_lookup.tsv` :

$$\text{consigne}(h) = \text{base} + \text{masque}(h) \times \text{magnitude}$$

`inject_setpoints(df, bldg_id)` ajoute deux colonnes en °C. Elle est appelée automatiquement
à chaque téléchargement (fonctions ci-dessous).

In [ ]:
import re, functools

META_PATH   = DATA_PROCESSED / 'metadata_clean.parquet'
LOOKUP_PATH = ROOT / 'data' / 'external' / 'options_lookup.tsv'   # depot NREL/resstock
COL_COOL    = 'out.schedules.cooling_setpoint..c'
COL_HEAT    = 'out.schedules.heating_setpoint..c'

def _f_abs_to_c(txt):    # '76F' -> 24.44  (temperature absolue)
    return (float(re.search(r'[-\d.]+', str(txt)).group()) - 32) * 5 / 9

def _f_delta_to_c(txt):  # '9F' -> 5.0  (ecart : PAS de -32)
    return float(re.search(r'[-\d.]+', str(txt)).group()) * 5 / 9

@functools.lru_cache(maxsize=1)
def _lookup_lines():
    return LOOKUP_PATH.read_text(encoding='utf-8').splitlines()

@functools.lru_cache(maxsize=None)
def _get_masks(characteristic, period):
    """(masque_semaine[24], masque_weekend[24]) signes -1/0/+1."""
    if period is None or str(period).strip().lower() == 'none':
        return tuple(np.zeros(24)), tuple(np.zeros(24))
    prefix = f'{characteristic}\t{period}\t'
    line = next((l for l in _lookup_lines() if l.startswith(prefix)), None)
    if line is None:
        raise ValueError(f'"{characteristic}" / "{period}" introuvable dans options_lookup')
    wk = re.search(r'weekday_setpoint_schedule=([-\d,\s]+)', line).group(1)
    we = re.search(r'weekend_setpoint_schedule=([-\d,\s]+)', line).group(1)
    arr = lambda s: tuple(int(x) for x in s.split(',')[:24])
    return arr(wk), arr(we)

@functools.lru_cache(maxsize=1)
def _setpoint_meta():
    cols = ['bldg_id',
            'in.cooling_setpoint', 'in.cooling_setpoint_has_offset',
            'in.cooling_setpoint_offset_magnitude', 'in.cooling_setpoint_offset_period',
            'in.heating_setpoint', 'in.heating_setpoint_has_offset',
            'in.heating_setpoint_offset_magnitude', 'in.heating_setpoint_offset_period']
    return pd.read_parquet(META_PATH, columns=cols).set_index('bldg_id')

def inject_setpoints(df, bldg_id):
    """Reconstruit et injecte les consignes chauffage + clim (colonnes en degC)."""
    ts = pd.DatetimeIndex(df['timestamp'] if 'timestamp' in df.columns else df.index)
    hours, wknd = ts.hour.values, (ts.dayofweek.values >= 5)
    row = _setpoint_meta().loc[bldg_id]
    for col, pfx, param in [
            (COL_COOL, 'in.cooling_setpoint', 'Cooling Setpoint Offset Period'),
            (COL_HEAT, 'in.heating_setpoint', 'Heating Setpoint Offset Period')]:
        has  = row[f'{pfx}_has_offset'] == 'Yes'
        base = _f_abs_to_c(row[pfx])
        mag  = _f_delta_to_c(row[f'{pfx}_offset_magnitude']) if has else 0.0
        wk, we = _get_masks(param, row[f'{pfx}_offset_period'] if has else None)
        m = np.where(wknd, np.asarray(we, float)[hours], np.asarray(wk, float)[hours])
        df[col] = base + m * mag

    # --- Arbitrage des consignes incohérentes -------------------------------------------
    # Guide technique ResStock 2025.1, p. 135, §8.4.5 :
    #   « If a sampled heating setpoint is greater than the cooling setpoint, the values are
    #     averaged and kept constant across heating and cooling seasons. »
    # « kept constant » = le MEME profil sert au chauffage et a la clim (constant D'UNE SAISON
    # A L'AUTRE), et non « constant dans le temps » : les deux schedules sont moyennes HEURE
    # PAR HEURE, offsets compris.
    # Verifie sur out.indoor_temperature.conditioned_space..c, au centieme de degre :
    #   4434   (75F sans offset / 65F sans offset) -> T_int 21.11 C constant   = (75+65)/2
    #   149091 (80F offset 3F 'Night +3h' / 70F)   -> T_int 23.89 C le jour    = (80+70)/2
    #                                                       23.06 C de 3h a 9h = (80-3+70)/2
    # Le declencheur porte sur les consignes ECHANTILLONNEES (les bases), pas sur le resultat
    # apres offset : un batiment dont les bases sont dans le bon ordre mais que le 'Night
    # Setback' clim fait passer sous la consigne de chauffage la nuit n'est PAS arbitre
    # (ex. 530669 : 75F / 76F). C'est ce que dit le guide, et ResStock le simule tel quel.
    # Concerne 108 des 503 batiments du parc.
    if _f_abs_to_c(row['in.heating_setpoint']) > _f_abs_to_c(row['in.cooling_setpoint']):
        moyenne = (df[COL_HEAT] + df[COL_COOL]) / 2
        df[COL_HEAT] = moyenne
        df[COL_COOL] = moyenne          # bande morte nulle : le systeme bascule en permanence
    return df

# Partie A — Timeseries individuelles par bâtiment

## 1. Filtrer les bâtiments (filtre du modèle)
Maison individuelle plein-pied, chauffage électrique, **sans VE / piscine / PV**, et **occupé**
(les vacants ont des schedules vides). Même périmètre que `lgbm_consommation_annuelle`.

In [3]:
raw = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=[
    'bldg_id', 'in.state', 'in.county', 'in.ashrae_iecc_climate_zone_2004',
    'in.geometry_building_type_recs', 'in.geometry_stories', 'in.heating_fuel',
    'in.electric_vehicle_ownership', 'in.misc_pool', 'in.has_pv', 'in.vacancy_status',
])

mask = (
    (raw['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (raw['in.geometry_stories'] == '1')                                  &
    (raw['in.heating_fuel'] == 'Electricity')                            &
    (raw['in.electric_vehicle_ownership'] == 'No')                       &
    (raw['in.misc_pool'] == 'None')                                      &
    (raw['in.has_pv'] == 'No')                                           &
    (raw['in.vacancy_status'] == 'Occupied')
)

candidates = raw[mask][['bldg_id', 'in.state', 'in.county',
                        'in.ashrae_iecc_climate_zone_2004']].reset_index(drop=True)
print(f'{len(candidates):,} bâtiments (filtre modèle + occupé)')
candidates.head(10)

55,732 bâtiments (filtre modèle + occupé)


,bldg_id,in.state,in.county,in.ashrae_iecc_climate_zone_2004
0,14,ID,G1600270,5B
1,15,FL,G1200310,2A
2,37,WV,G5400350,4A
3,44,TX,G4800930,3A
4,49,WA,G5300530,4C
5,58,AR,G0500050,4A
6,62,OR,G4100510,4C
7,76,MS,G2800110,3A
8,77,TX,G4804890,2A
9,79,AZ,G0400130,2B


## 2. Télécharger la timeserie d'un bâtiment
Les fichiers sont publics sur OEDI — aucune clé AWS requise.  
URL : `{OEDI_BASE}/state={state}/{bldg_id}-0.parquet`

In [4]:
def download_timeseries(bldg_id: int, state: str, save: bool = True) -> pd.DataFrame:
    url = f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet'
    print(f'Téléchargement : {url}')
    df = pd.read_parquet(url)
    df = inject_setpoints(df, bldg_id)          # consignes reconstruites (chauffage + clim)
    if save:
        out = DATA_PROCESSED / f'{bldg_id}-0.parquet'
        df.to_parquet(out)
        print(f'Sauvegardé : {out}')
    print(f'Shape : {df.shape} | {df["timestamp"].iloc[0]} → {df["timestamp"].iloc[-1]}')
    return df

In [5]:
# Exemple : le premier bâtiment candidat
row   = candidates.iloc[0]
df_ts = download_timeseries(int(row['bldg_id']), row['in.state'])
df_ts.head()

Téléchargement : https://oedi-data-lake.s3.amazonaws.com/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2025/resstock_amy2018_release_1/timeseries_individual_buildings/by_state/upgrade=0/state=ID/14-0.parquet
Sauvegardé : C:\Users\bamdyoun\stage\FlexiMax\data\processed\14-0.parquet
Shape : (35040, 192) | 2018-01-01 00:15:00 → 2019-01-01 00:00:00


,bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,...,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.peak_period,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy
0,14,2018-01-01 00:15:00,1698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.3000,0.0,0.0,1.0,NaN,0.670,0.2760,0.0,NaN,0.0
1,14,2018-01-01 00:30:00,1698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.3000,0.0,0.0,1.0,NaN,0.670,0.2760,0.0,NaN,0.0
2,14,2018-01-01 00:45:00,1698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.2670,0.0,0.0,1.0,NaN,0.656,0.2410,0.0,NaN,0.0
3,14,2018-01-01 01:00:00,1698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0491,0.0,0.0,1.0,NaN,0.562,0.0159,0.0,NaN,0.0
4,14,2018-01-01 01:15:00,1698.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0491,0.0,0.0,1.0,NaN,0.562,0.0159,0.0,NaN,0.0


In [ ]:
182255 in candidates["bldg_id"].values

In [ ]:
# Exemple : bldg_id 347201 — Texas, zone 3A
bldg_id = 347201
state   = candidates.loc[candidates['bldg_id'] == bldg_id, 'in.state'].values[0]

df_ts = download_timeseries(bldg_id, state)
df_ts.head()



In [ ]:
bldg_id = 347201
state = "VA"

df_ts = download_timeseries(bldg_id, state)

## 3. Télécharger un lot pour le réseau (échantillon stratifié par zone)
Lecture **par colonnes** (météo + schedules + cibles) → ~1 Mo/bâtiment au lieu de 9,3.
Échantillon réparti sur les zones climatiques.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pyarrow.parquet as pq

# Colonnes utiles au réseau (parquet colonnaire -> ~1 Mo/bâtiment au lieu de 9,3)
WEA = ['out.outdoor_air_drybulb_temp..c', 'out.outdoor_air_relative_humidity..percentage',
       'out.weather.wind_speed..meter_per_second',
       'out.weather.direct_normal_solar_radiation..watt_per_m2',
       'out.weather.diffuse_solar_radiation..watt_per_m2']
SCHED = ['out.schedules.' + s for s in [
    'occupants', 'vacancy', 'lighting_interior', 'lighting_garage', 'plug_loads_other',
    'plug_loads_tv', 'clothes_dryer', 'clothes_washer', 'dishwasher', 'cooking_range',
    'ceiling_fan', 'hot_water_fixtures', 'hot_water_clothes_washer', 'hot_water_dishwasher',
    'no_space_cooling', 'no_space_heating']]
TGT = ['out.electricity.' + t + '.energy_consumption..kwh' for t in
       ['total', 'heating', 'cooling', 'hot_water']]
NN_COLS = ['timestamp'] + WEA + SCHED + TGT

# Les 2 consignes ne sont pas téléchargées (vides à la source) : inject_setpoints les CRÉE.
# Ce sont elles que le réseau doit retrouver dans le fichier final -> 28 colonnes au total.
SETP = [COL_COOL, COL_HEAT]

# Sur les 23 out.schedules.* de la source, 7 sont inutilisables dans cette release :
# 100 % NaN (cooling_setpoint, heating_setpoint, electric_vehicle_charging/discharging,
# peak_period, pre_peak_period) ou constant à 0 (power_outage). Les 16 de SCHED sont les
# seuls porteurs d'information — d'où ce sous-ensemble et non les 23.

def download_nn(bldg_id, state, force=False):
    """Télécharge les colonnes utiles d'un bâtiment et y injecte les consignes reconstruites."""
    out = DATA_PROCESSED / f'{bldg_id}-0.parquet'
    if out.exists() and not force:
        return
    df = pd.read_parquet(f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet', columns=NN_COLS)
    df = inject_setpoints(df, bldg_id)
    assert set(SETP) <= set(df.columns), f'consignes non injectées pour {bldg_id}'
    df.to_parquet(out)

# Échantillon stratifié par zone climatique (>= 1 par zone, ~N au total)
N, zcol = 500, 'in.ashrae_iecc_climate_zone_2004'
parts = [g.sample(min(len(g), max(1, round(N * len(g) / len(candidates)))), random_state=42)
         for _, g in candidates.groupby(zcol)]
sample = pd.concat(parts).reset_index(drop=True)
print(f'{len(sample)} bâtiments à télécharger sur {sample[zcol].nunique()} zones')

# FORCE = True : re-télécharge même si le fichier local existe. Nécessaire après toute
# modification de NN_COLS ou de inject_setpoints, sinon les anciens fichiers sont conservés
# tels quels et la modification reste invisible.
FORCE = False

jobs = list(zip(sample['bldg_id'].astype(int), sample['in.state']))
def _get(job):
    bid, st = job
    try:
        download_nn(bid, st, force=FORCE)
    except Exception as e:
        return f'  skip {bid} : {e}'

with ThreadPoolExecutor(max_workers=8) as ex:      # ~1-2 min
    for msg in ex.map(_get, jobs):
        if msg:
            print(msg)

# Contrôle : aucun fichier ne doit sortir sans ses consignes.
# Sans ce test, un lot produit par une définition périmée de download_nn (kernel non
# redémarré) passe inaperçu — les fichiers existent, mais amputés de 2 colonnes.
sans = [int(b) for b in sample['bldg_id']
        if not set(SETP) <= set(pq.ParquetFile(DATA_PROCESSED / f'{b}-0.parquet').schema.names)]
print(f'fichiers avec consignes : {len(sample) - len(sans)} / {len(sample)}')
assert not sans, (
    f'{len(sans)} fichier(s) sans consignes, ex. {sans[:5]}.\n'
    "-> Redémarrer le kernel, réexécuter les cellules dans l'ordre, et mettre FORCE = True "
    'pour écraser les fichiers déjà présents.')

sample[['bldg_id', 'in.state']].to_csv(DATA_PROCESSED / 'nn_buildings.csv', index=False)
print('terminé ->', DATA_PROCESSED / 'nn_buildings.csv')

153 bâtiments à télécharger sur 16 zones
fichiers avec consignes : 153 / 153
terminé -> C:\Users\bamdyoun\stage\FlexiMax\data\processed\nn_buildings.csv


# Partie B — Météo statique par comté

## 4. Agrégats climatiques
Un CSV météo léger par comté (= la météo exacte de simulation, validée : écart ~0). On en extrait des grandeurs statiques qui couplent avec les agrégats d'enveloppe — `HDD/CDD` × `UA`, `GHI` × `A_solaire`, `vent` × `H_ve`. Jointure sur `in.county`.  
URL : `{WEATHER_BASE}/state={state}/{county}_2018.csv`

In [7]:
def weather_static(county: str, state: str, base: float = 18.0) -> dict:
    """Agrégats climatiques annuels d'un comté (CSV météo horaire OEDI)."""
    w = pd.read_csv(f'{WEATHER_BASE}/state={state}/{county}_2018.csv', parse_dates=['date_time'])
    T, ghi = w['Dry Bulb Temperature [°C]'], w['Global Horizontal Radiation [W/m2]']
    hiver = w['date_time'].dt.month.isin([12, 1, 2])
    return {
        'in.county'   : county,
        'HDD18'       : np.maximum(base - T, 0).sum() / 24,   # °C·jour chauffage
        'CDD18'       : np.maximum(T - base, 0).sum() / 24,   # °C·jour clim
        'T_moy'       : T.mean(),
        'T_design_min': T.quantile(0.01),
        'T_design_max': T.quantile(0.99),
        'GHI_an'      : ghi.sum() / 1000,                     # kWh/m²/an
        'GHI_hiver'   : ghi[hiver].sum() / 1000,
        'vent_moy'    : w['Wind Speed [m/s]'].mean(),
        'RH_moy'      : w['Relative Humidity [%]'].mean(),
    }

# test sur le comté du bâtiment 159
weather_static('G4804010', 'TX')

{'in.county': 'G4804010',
 'HDD18': np.float64(1316.34),
 'CDD18': np.float64(1700.4054166666665),
 'T_moy': np.float64(19.05223401826484),
 'T_design_min': np.float64(-3.9),
 'T_design_max': np.float64(36.7),
 'GHI_an': np.float64(1672.9745),
 'GHI_hiver': np.float64(224.5935),
 'vent_moy': np.float64(3.2155844748858446),
 'RH_moy': np.float64(75.86168607305936)}

In [ ]:
# Batch : 1 fichier météo par comté (partagé par tous ses bâtiments) -> weather_static.parquet
counties = (candidates[['in.county', 'in.state']]
            .drop_duplicates()
            .loc[lambda d: d['in.county'].str.startswith('G')])   # exclut AK/HI (non-GISJOIN)

rows = []
for _, r in counties.iterrows():
    try:
        rows.append(weather_static(r['in.county'], r['in.state']))
    except Exception as e:
        print(f'  skip {r["in.county"]} : {e}')

weather = pd.DataFrame(rows)
weather.to_parquet(DATA_PROCESSED / 'weather_static.parquet')
print(f'{len(weather)} comtés → weather_static.parquet')
weather.head()